In [ ]:
#Create the GDF for the reanalysis/observations using this jupyter notebook 

In [26]:
import numpy as np
import xarray as xr
import sys

In [48]:
SaveData='/bettik/PROJECTS/pr-regional-climate/fainx/'
sourceData='/bettik/PROJECTS/pr-regional-climate/fainx/'

domain = 'GRf'

#Full HMA
lat_min = 45
lat_max = 20

lon_min = 60
lon_max = 110

###########################################################
# var='t2m'
# model='era5'
# fileName='era5_'+domain+'_t2m.nc'
# ds0= xr.open_dataset(sourceData+model+'/'+fileName)
# print(ds0)
# print(ds0.coords)
# ds0 = ds0.rename({'latitude': 'lat', 'longitude':'lon'})

############################################################
var='t2m'
model='era5-land'
fileName = 'era5_land_'+domain+'_t2m.nc'  
ds0= xr.open_dataset(sourceData+model+'/'+fileName)
ds0 = ds0.rename({'latitude': 'lat', 'longitude':'lon'})
print(ds0)
print(ds0.coords)

# ###########################################################
# sourceData='/bettik/PROJECTS/pr-regional-climate/fainx/'
# var='tmp'
# model='cru'
# fileName='cru_ts4.05.1901.2020.tmp.dat_HMA.nc'
# ds0= xr.open_dataset(sourceData+model+'/'+fileName)
# print(ds0)
# print(ds0.coords)

# ###########################################################

<xarray.Dataset>
Dimensions:     (valid_time: 304, latitude: 25, longitude: 49)
Coordinates:
    number      int64 ...
  * valid_time  (valid_time) datetime64[ns] 2000-01-01 2000-02-01 ... 2025-04-01
  * latitude    (latitude) float64 42.0 41.75 41.5 41.25 ... 36.5 36.25 36.0
  * longitude   (longitude) float64 66.0 66.25 66.5 66.75 ... 77.5 77.75 78.0
    expver      (valid_time) object ...
Data variables:
    z           (valid_time, latitude, longitude) float32 ...
    sdor        (valid_time, latitude, longitude) float32 ...
    t2m         (valid_time, latitude, longitude) float32 ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
Coordinates:
    number      int64 ...
  * valid_time  (valid_time) datetime64[ns] 2000-01-01 2000-02-01 ... 2025-04-01
  * latitu

In [49]:
print("It is only needed to change name of variable and eventually,the name of coordinates:\n var_obs.latitude to var_obs.lat")
print("NOTE: write x,y in minus if not it cannot read cdo")

It is only needed to change name of variable and eventually,the name of coordinates:
 var_obs.latitude to var_obs.lat
NOTE: write x,y in minus if not it cannot read cdo


In [50]:
var_obs=ds0[var]
lat2d, lon2d = np.meshgrid(var_obs.lat, var_obs.lon)

lat2D = xr.DataArray(data=lat2d.transpose(),  dims=["y", "x"],
    coords=dict(x=(["x"], var_obs.lon.values), y=(["y"], var_obs.lat.values)), name='lat')

lon2D = xr.DataArray(data=lon2d.transpose(),  dims=["y", "x"],
    coords=dict(x=(["x"], var_obs.lon.values), y=(["y"], var_obs.lat.values)), name='lon')
lonlat = xr.merge([lat2D, lon2D])
lonlat.to_netcdf(SaveData+model+'/'+'grid_'+var+'_'+model+'_2d_undef_'+domain+'.nc')

In [51]:
##

In [52]:
##Opening the undefined grid to add dummy variables and attributes 
ds=xr.open_dataset(SaveData+model+'/'+'grid_'+var+'_'+model+'_2d_undef_'+domain+'.nc')
#ds
print('grid_'+var+'_'+model+'_2d_undef_'+domain+'.nc')

grid_t2m_era5_2d_undef_GRf.nc


In [53]:
# Create dummy variable based on lat (or any constant field; as in make_gdf.sh C.Amory)
dummy = xr.DataArray(
    np.ones_like(ds['lat']),
    dims=("y", "x"),
    coords={"y": ds["y"], "x": ds["x"]},
    name="dummy"
)

# Assign required attributes
dummy.attrs = {
    "long_name": "dummy variable",
    "standard_name": "latitude",  
    "units": "1",
    "valid_range": (-1.e+20, 1.e+20),
    "coordinates": "lon lat"
}

# Lat and lon attributes
ds["lat"].attrs = {
    "units": "degrees_north",
    "long_name": "grid center latitude",
    "standard_name": "latitude",
    "valid_range": (-1.e+20, 1.e+20),
    "actual_range": (float(ds["lat"].min()), float(ds["lat"].max()))
}

ds["lon"].attrs = {
    "units": "degrees_east",
    "long_name": "grid center longitude",
    "standard_name": "longitude",
    "valid_range": (-1.e+20, 1.e+20),
    "actual_range": (float(ds["lon"].min()), float(ds["lon"].max()))
}

# opt: set x/y metadata
ds["x"].attrs = {
    "units": "km",
    "long_name": "X",
    "standard_name": "X"
}
ds["y"].attrs = {
    "units": "km",
    "long_name": "Y",
    "standard_name": "Y"
}

# Add the dummy variable to the dataset
ds["dummy"] = dummy

# Set global attributes
ds.attrs = {
    "title": "Grid file generated for CDO remapping - MAR model",
    "institution": "Generated via Python meshgrid",
    "history": "Created for remapping using cdo remapbil: "
}

# Save to new file
#ds.to_netcdf(sourceData+model+'/'+'grid_'+var+'_'+model+'_2d_def.nc')
ds.to_netcdf(SaveData+model+'/'+'grid_'+model+'_2d_def_'+domain+'.nc')
print(SaveData+model+'/'+'grid_'+var+'_'+model+'_2d_def_'+domain+'.nc')

/bettik/PROJECTS/pr-regional-climate/fainx/era5/grid_t2m_era5_2d_def_GRf.nc


In [25]:
#ncdump -h /bettik/PROJECTS/pr-regional-climate/santolam/cru/grid_tmp_cru_2d_def.nc